# OpenAgent Prompt System Demo

This notebook demonstrates the three layers of the `openagent.prompts` module:

1. **Content** — `load()` / `find()` for read-only `.md` fragment text
2. **Sections** — self-contained functions that produce prompt sections
3. **Compose** — `compose()` joins sections via ordered profiles

Each layer is independent and composable. You can use any layer directly.

In [ ]:
import sys
sys.path.insert(0, "../libs/openagent")

## Layer 1: Content — Loading `.md` Fragments

The content layer is a read-only loader. It scans the `prompts/` package for `.md` files
and exposes them by key (filename minus `.md` extension). Results are cached.

In [ ]:
from openagent.prompts import load, find

### Discover available fragments with `find()`

In [ ]:
# Find all system prompt fragments
system_fragments = find("system-prompt-")
print(f"System prompt fragments ({len(system_fragments)}):")
for key in system_fragments:
    print(f"  - {key}")

In [ ]:
# Find all tool instruction fragments
tool_fragments = find("tool-instruction-")
print(f"Tool instruction fragments ({len(tool_fragments)}):")
for key in tool_fragments:
    print(f"  - {key}")

In [ ]:
# Find user-facing prompt fragments (used by middleware, not system prompt)
user_fragments = find("user-prompt-")
print(f"User prompt fragments ({len(user_fragments)}):")
for key in user_fragments:
    print(f"  - {key}")

### Load a specific fragment with `load()`

In [ ]:
# Load and display the identity fragment
identity_text = load("system-prompt-identity")
print(identity_text[:500])

In [ ]:
# Fragments with ${var} placeholders — these are resolved at runtime by section functions
git_template = load("system-prompt-git-status")
print("Git status template (note the ${var} placeholders):")
print(git_template)

In [ ]:
# Missing keys raise KeyError
try:
    load("nonexistent-fragment")
except KeyError as e:
    print(f"KeyError: {e}")

## Layer 2: Sections — Self-Contained Prompt Functions

Each section function has the signature `(PromptContext) -> str | None`.
It loads its own content, resolves its own variables, and returns `None` to opt out.

There is no global variable dictionary — each section is independent.

In [ ]:
from openagent.prompts.sections import (
    PromptContext,
    GitContext,
    identity,
    doing_tasks,
    tool_usage_policy,
    tool_instructions,
    skills,
    environment,
    git_status,
    user_instructions,
)

### Sections that are always included

In [ ]:
# identity() always returns content regardless of context
ctx = PromptContext()  # empty context
result = identity(ctx)
print(f"identity() returned {len(result)} chars")
print(result[:200], "...")

### Sections that opt out when data is missing

In [ ]:
# tool_usage_policy() returns None when no tools are registered
ctx_empty = PromptContext()
print(f"tool_usage_policy(no tools): {tool_usage_policy(ctx_empty)}")
print(f"skills(no skills):           {skills(ctx_empty)}")
print(f"git_status(no git):          {git_status(ctx_empty)}")
print(f"user_instructions(none):     {user_instructions(ctx_empty)}")

### Sections with runtime data

In [ ]:
from openagent.types import Skill, MCPServer

# Skills section
ctx_with_skills = PromptContext(
    skills=[
        Skill(name="commit", description="Create git commits", path="/skills/commit"),
        Skill(name="review", description="Review code changes", path="/skills/review"),
    ]
)
print(skills(ctx_with_skills))

In [ ]:
# Environment section
ctx_with_env = PromptContext(
    environment={
        "Platform": "darwin",
        "Python": "3.11",
        "Working Directory": "/home/user/project",
    }
)
print(environment(ctx_with_env))

In [ ]:
# Git status section — resolves ${var} placeholders from GitContext
ctx_with_git = PromptContext(
    git=GitContext(
        current_branch="feat/agent-loop",
        main_branch="main",
        status="M openagent/prompts/sections.py\nM openagent/prompts/__init__.py",
        recent_commits="abc1234 refactor: redesign prompt system\ndef5678 feat: add tool instructions",
    )
)
print(git_status(ctx_with_git))

In [ ]:
# User instructions section
ctx_with_instructions = PromptContext(
    user_instructions="Always write tests before implementation. Use Google-style docstrings."
)
print(user_instructions(ctx_with_instructions))

### Tool instructions with mock tools

The `tool_instructions()` section automatically:
- Loads `tool-instruction-{name}.md` for each tool
- Appends supplementary fragments (e.g. `tool-instruction-bash-git-*`)
- Falls back to `tool.instruction` for tools without `.md` files
- Resolves `${tool_*}` cross-references and date variables

In [ ]:
from pydantic import BaseModel
from openagent.tools.base import BaseAgentTool
from openagent.types import ToolResult


class NoParams(BaseModel):
    pass


# Minimal mock tools — only `name` matters for prompt assembly
class BashTool(BaseAgentTool[NoParams]):
    name: str = "bash"
    args_schema = NoParams
    async def execute(self, params: NoParams) -> ToolResult:
        return ToolResult(output="")


class ReadTool(BaseAgentTool[NoParams]):
    name: str = "read"
    args_schema = NoParams
    async def execute(self, params: NoParams) -> ToolResult:
        return ToolResult(output="")


class GlobTool(BaseAgentTool[NoParams]):
    name: str = "glob"
    args_schema = NoParams
    async def execute(self, params: NoParams) -> ToolResult:
        return ToolResult(output="")

In [ ]:
ctx_with_tools = PromptContext(tools=[BashTool(), ReadTool(), GlobTool()])
result = tool_instructions(ctx_with_tools)

# Show the structure (first 1500 chars)
print(result[:1500])
print(f"\n... ({len(result)} total chars)")

In [ ]:
# Verify that ${tool_*} cross-references were resolved
assert "${tool_bash}" not in result, "tool_bash was not resolved!"
assert "${tool_read}" not in result, "tool_read was not resolved!"
assert "${tool_glob}" not in result, "tool_glob was not resolved!"
assert "${git_parallel_note}" not in result, "git_parallel_note was not resolved!"
print("All ${var} placeholders resolved successfully.")

### Custom tools with inline instructions

Third-party tools that don't have a `.md` file can carry instructions
via the `instruction` field on `BaseAgentTool`.

In [ ]:
class DatabaseTool(BaseAgentTool[NoParams]):
    name: str = "database"
    instruction: str = (
        "Execute SQL queries against the project database.\n\n"
        "Usage:\n"
        "- Only SELECT queries are allowed by default\n"
        "- Use parameterized queries to prevent SQL injection\n"
        "- Results are returned as JSON arrays"
    )
    args_schema = NoParams
    async def execute(self, params: NoParams) -> ToolResult:
        return ToolResult(output="")


ctx_custom = PromptContext(tools=[BashTool(), DatabaseTool()])
result = tool_instructions(ctx_custom)

# Show just the database tool section
db_start = result.index("## database")
print(result[db_start:])

## Layer 3: Compose — Profiles and Assembly

Profiles are ordered lists of section functions. `compose()` calls each one,
filters out `None` returns, and joins the results with `\n\n`.

In [ ]:
from openagent.prompts import compose, FRESH_SESSION, sections

### Using the built-in `FRESH_SESSION` profile

In [ ]:
# Minimal context — only required sections are included
minimal_ctx = PromptContext()
prompt = compose(FRESH_SESSION, minimal_ctx)

print(f"Minimal prompt: {len(prompt)} chars")
print()

# Show which headings are present
for line in prompt.split("\n"):
    if line.startswith("# "):
        print(f"  {line}")

In [ ]:
# Full context — all sections included
full_ctx = PromptContext(
    tools=[BashTool(), ReadTool(), GlobTool()],
    skills=[
        Skill(name="commit", description="Create git commits", path="/skills/commit"),
        Skill(name="review", description="Review code changes", path="/skills/review"),
    ],
    mcps=[MCPServer(name="context7", description="Documentation lookup")],
    environment={"Platform": "darwin", "Python": "3.11"},
    git=GitContext(
        current_branch="feat/agent-loop",
        main_branch="main",
        status="M openagent/prompts/sections.py",
        recent_commits="abc1234 refactor: redesign prompt system",
    ),
    scratchpad_dir="/tmp/openagent-scratch",
    user_instructions="Be concise. Prefer simple solutions.",
)
prompt = compose(FRESH_SESSION, full_ctx)

print(f"Full prompt: {len(prompt)} chars")
print()

# Show all headings
for line in prompt.split("\n"):
    if line.startswith("# "):
        print(f"  {line}")

### Custom profiles

Profiles are just lists of section functions. You can create your own
with any subset or ordering.

In [ ]:
# A lightweight profile for quick tasks — skip detailed tool instructions
LIGHTWEIGHT = [
    sections.identity,
    sections.tone_and_style,
    sections.environment,
    sections.user_instructions,
]

ctx = PromptContext(
    environment={"Platform": "linux"},
    user_instructions="Answer briefly.",
)
prompt = compose(LIGHTWEIGHT, ctx)
print(f"Lightweight prompt: {len(prompt)} chars")
print()
for line in prompt.split("\n"):
    if line.startswith("# "):
        print(f"  {line}")

### Custom section functions

Any callable matching `(PromptContext) -> str | None` can be a section.
You can write your own to inject domain-specific instructions.

In [ ]:
def safety_rules(_ctx: PromptContext) -> str | None:
    """Custom section: safety rules for production environments."""
    return (
        "# Safety Rules\n\n"
        "- Never execute destructive operations without confirmation\n"
        "- Always create backups before modifying databases\n"
        "- Log all actions to the audit trail"
    )


def project_context(ctx: PromptContext) -> str | None:
    """Custom section: include only when we have git info."""
    if ctx.git is None:
        return None
    return (
        f"# Project Context\n\n"
        f"You are working on branch `{ctx.git.current_branch}`.\n"
        f"The main branch is `{ctx.git.main_branch}`."
    )


# Mix built-in and custom sections
PRODUCTION_PROFILE = [
    sections.identity,
    safety_rules,           # custom
    project_context,        # custom, conditional
    sections.tool_instructions,
    sections.user_instructions,
]

ctx = PromptContext(
    tools=[BashTool()],
    git=GitContext(
        current_branch="hotfix/urgent",
        main_branch="main",
        status="",
        recent_commits="",
    ),
    user_instructions="This is a production hotfix. Be extra careful.",
)
prompt = compose(PRODUCTION_PROFILE, ctx)

for line in prompt.split("\n"):
    if line.startswith("# "):
        print(f"  {line}")

# Show the custom sections
print()
safety_start = prompt.index("# Safety Rules")
project_start = prompt.index("# Project Context")
tools_start = prompt.index("# Tools")
print(prompt[safety_start:tools_start])

## User-Facing Prompts (Compaction)

The `user-prompt-*` fragments are not part of the system prompt.
They are used by the middleware for compaction (context compression).

- `user-prompt-compaction-request` — sent to the LLM to request a summary
- `user-prompt-compaction-summary-rebuild` — template for rebuilding context after compaction

In [ ]:
# Compaction request prompt (sent as-is to the LLM)
print("=== Compaction Request ===")
print(load("user-prompt-compaction-request"))

In [ ]:
# Summary rebuild template (${summary_content} replaced by middleware)
template = load("user-prompt-compaction-summary-rebuild")
print("=== Summary Rebuild Template ===")
print(template)
print()

# How the middleware uses it:
rebuilt = template.replace("${summary_content}", "User asked to refactor the auth module. Files modified: auth.py, middleware.py.")
print("=== After substitution ===")
print(rebuilt)

## Architecture Recap

```
prompts/
  content.py          load(), find()     # read-only .md loader
  sections.py         12 section fns     # each: (PromptContext) -> str | None
  __init__.py         compose(), profiles # composition layer
  system-prompt-*.md                      # system prompt fragments
  tool-instruction-*.md                   # tool usage instructions
  user-prompt-*.md                        # user-facing prompts (compaction)
```

Key design principles:
- **No global variable dictionary** — each section resolves its own vars
- **No `string.Template`** — `$` is always literal in `.md` files
- **Sections opt out by returning `None`** — no conditional logic in the assembler
- **Profiles are just lists** — create custom profiles by mixing sections
- **Third-party tools use `instruction` field** — no `.md` file required